# 综合实战：全国地图可视化

---

**这个案例把前面学的全用上了**

```
定位文件 → 读 JSON → 判断结构 → 提取数据 → 转成 (地名, 数值) → 画地图 → 导出 HTML
```

**学习目标**

1. 处理真实世界那种"套了好几层"的 JSON 数据
2. 用列表推导式把嵌套数据"拍平"成绘图需要的格式
3. 掌握分段型视觉映射在"数值跨度大"场景下的必要性
4. 学会"先探索、再取数"的工作方法

**数据说明**

原始笔记读的是桌面资料目录（已不存在）。现在的数据在本目录的
`data/地图数据/疫情.txt`，结构与原数据一致，内容为**合成的练习数据**。
数字不代表真实统计，仅用于演示画图方法。


---
## 一、定位数据文件


In [ ]:
import json
from pathlib import Path


def data_path(rel):
    """在若干个候选目录下寻找 data/<rel>，返回第一个存在的路径"""
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / '09.数据可视化',
        cwd / 'src' / 'pythonlearnproject' / '09.数据可视化',
        cwd.parent / '09.数据可视化',
        cwd.parent,
    ]
    for base in candidates:
        p = base / 'data' / rel
        if p.exists():
            return p
    raise FileNotFoundError(f'找不到 data/{rel}，当前目录是 {cwd}')


print('数据目录：', data_path('地图数据'))


---
## 二、读文件 + 解析 JSON

用 `with` 语句，编码明确写 `utf-8`。

**注意**：这份数据是**纯 JSON**，没有 JSONP 外壳，所以可以直接 `json.loads`。
但"先看一下开头"这个习惯要保持 —— 这是区分纯 JSON 和 JSONP 的唯一方法。


In [ ]:
source = data_path('地图数据/疫情.txt')

with open(source, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('文件长度：', len(raw_text))
print('开头 60 个字符：', raw_text[:60])
print('是以 { 开头吗？', raw_text.strip().startswith('{'), ' -> 说明是纯 JSON，可以直接解析')

data_dict = json.loads(raw_text)
print('顶层键：', list(data_dict.keys()))


---
## 三、探索结构：一层一层往里走

原始笔记里直接写了 `data_dict["areaTree"][0]["children"]`，
但如果不先看清结构，这个路径是"猜"出来的。

**正确做法是逐层打印**：


In [ ]:
print('第 1 层：data_dict 的键 =', list(data_dict.keys()))

area_tree = data_dict['areaTree']
print(f'第 2 层：areaTree 是 list，长度 {len(area_tree)}')
print('        第 0 个元素的键 =', list(area_tree[0].keys()))
print('        第 0 个元素的名字 =', area_tree[0]['name'])

children = area_tree[0]['children']
print(f'第 3 层：children 是 list，长度 {len(children)}  <- 每个元素是一个省级行政区')
print('        第 0 个元素 =', children[0])
print('        它的键 =', list(children[0].keys()))

print()
print('看前 5 个省级行政区：')
for item in children[:5]:
    print(f"  {item['name']}: {item['total']['confirm']}")


结构清楚了，路径就是：

```
data_dict
 └─ 'areaTree'                    (list)
     └─ [0]                       (dict, 国家层)
         └─ 'children'            (list, 各省)
             └─ [n]['name']           省份名
             └─ [n]['total']['confirm']  确诊人数
```


---
## 四、提取数据：列表推导式一行搞定

目标格式是 `[(省份名, 数值), ...]`，这是 pyecharts 地图要求的输入格式。


In [ ]:
# 写法 1：普通 for 循环（直观，好理解）
data_list = []
for province in children:
    name = province['name']
    confirm = province['total']['confirm']
    data_list.append((name, confirm))

print('写法 1 得到', len(data_list), '条数据，前 3 条：', data_list[:3])

# 写法 2：列表推导式（等价，更简洁）
data_list2 = [(p['name'], p['total']['confirm']) for p in children]
print('写法 2 结果一致吗？', data_list == data_list2)


> **原始笔记的修正说明**
>
> 原代码里有一句：
>
> ```python
> from operator import concat
> province_name = concat(province_data["name"], "省")   # 给每个名字加"省"
> ```
>
> **这是错的。** `operator.concat("湖北", "省")` 确实会得到 `"湖北省"`，
> 但 pyecharts 的地图匹配用的是**不带后缀的标识名**，加完后缀就对不上了：
>
> - `北京市省`、`内蒙古省`、`香港省` …… 全都匹配失败
> - 地图上这些区域会**保持灰色，而且不报任何错**（最难查的那种 bug）
>
> **正确做法：直接用数据里原来的名字，不做任何拼接。**
> 这也是"代码能跑"和"结果正确"之间的差距 —— 一定要盯着输出检查。


In [ ]:
# 检查一下：有没有哪个地名加了后缀（自查习惯）
suffixes = ('省', '市', '自治区', '特别行政区')
suspicious = [name for name, _ in data_list if name.endswith(suffixes)]
print('疑似带后缀的地名：', suspicious if suspicious else '无，很好')
print('前 5 个地名：', [name for name, _ in data_list[:5]])

# 顺便看一眼数值范围 —— 这决定了后面该用哪种视觉映射
values = [v for _, v in data_list]
print(f'数值范围：{min(values)} ~ {max(values)}')
print(f'最大值所在的省：{max(data_list, key=lambda x: x[1])}')
print('跨度这么大，说明应该用【分段型】而不是连续型视觉映射')


---
## 五、画图

### 5.1 先用连续型看看效果（会发现不理想）


In [ ]:
from pyecharts.charts import Map
from pyecharts import options as opts

map_cont = Map()
map_cont.add('确诊人数', data_list, 'china')
map_cont.set_global_opts(
    title_opts=opts.TitleOpts(title='连续型视觉映射的问题演示', pos_left='center'),
    visualmap_opts=opts.VisualMapOpts(min_=0, max_=max(values)),
)
map_cont.render('全国疫情_连续型.html')
print('已生成 全国疫情_连续型.html —— 打开看看，绝大部分省份都是同一种浅色')
# 原因：最大值 6 万多，而大部分省份在几百到几千，颜色差距被压缩得看不出来


### 5.2 改用分段型（正式版本）


In [ ]:
map_chart = Map()
map_chart.add(
    series_name='各省份确诊人数',
    data_pair=data_list,
    maptype='china',
    is_map_symbol_show=False,           # 不显示红色小圆点，纯靠颜色表达
    label_opts=opts.LabelOpts(is_show=False),   # 地名标签也关掉，避免遮挡
)

map_chart.set_global_opts(
    title_opts=opts.TitleOpts(
        title='全国疫情分布（练习数据）',
        subtitle='数据为合成的练习数据，非真实统计',
        pos_left='center',
        pos_top='2%',
    ),
    legend_opts=opts.LegendOpts(is_show=False),
    toolbox_opts=opts.ToolboxOpts(is_show=True),
    tooltip_opts=opts.TooltipOpts(formatter='{b}<br/>确诊：{c} 人'),
    visualmap_opts=opts.VisualMapOpts(
        is_show=True,
        is_piecewise=True,              # 关键：分段模式
        pos_left='5%',
        pos_bottom='15%',
        pieces=[
            # 每一档写清楚 min / max / label / color
            # 注意颜色值只有一个 #，原始笔记里的 "##0000FF" 是多写了一个的笔误
            {'min': 0,    'max': 99,   'label': '0-99 人',       'color': '#EAF6FF'},
            {'min': 100,  'max': 499,  'label': '100-499 人',    'color': '#A8D8FF'},
            {'min': 500,  'max': 999,  'label': '500-999 人',    'color': '#5AA9E6'},
            {'min': 1000, 'max': 1999, 'label': '1000-1999 人',  'color': '#3B7DD8'},
            {'min': 2000, 'max': 9999, 'label': '2000-9999 人',  'color': '#1F4E9C'},
            {'min': 10000,             'label': '10000 人以上',  'color': '#8B0000'},
        ],
    ),
)

map_chart.render('全国疫情地图.html')
print('已生成 全国疫情地图.html')


### 5.3 加一个"排行榜"作为补充

地图擅长表达"空间分布"，但**不擅长精确比较大小**。
想知道"到底哪几个省最多"，配一张横向条形图最清楚。


In [ ]:
from pyecharts.charts import Bar

# 取前 10 名，按数值升序排（横向图里大的会显示在顶部）
top10 = sorted(data_list, key=lambda x: x[1], reverse=True)[:10]
top10_asc = sorted(top10, key=lambda x: x[1])

bar = Bar()
bar.add_xaxis([name for name, _ in top10_asc])
bar.add_yaxis(
    '确诊人数',
    [v for _, v in top10_asc],
    label_opts=opts.LabelOpts(is_show=True, position='right'),
)
bar.reversal_axis()                     # 变成横向条形图
bar.set_global_opts(
    title_opts=opts.TitleOpts(title='确诊人数 TOP 10', pos_left='center'),
    xaxis_opts=opts.AxisOpts(name='人数'),
    legend_opts=opts.LegendOpts(is_show=False),
    tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='shadow'),
)
bar.render('全国疫情TOP10.html')
print('已生成 全国疫情TOP10.html')


---
## 六、把结果导出成 CSV（方便给别人）

画完图常常还要把**处理后的数据**交给别人，用 `csv` 模块几行就能写。


In [ ]:
import csv

out_file = Path('全国疫情统计_导出.csv')
with open(out_file, 'w', encoding='utf-8-sig', newline='') as f:
    # utf-8-sig 会写入 BOM，这样用 Excel 打开中文不会乱码
    writer = csv.writer(f)
    writer.writerow(['省份', '确诊人数'])               # 表头
    writer.writerows(sorted(data_list, key=lambda x: x[1], reverse=True))

print('已导出：', out_file.resolve())
print('用 Excel 打开看看中文有没有乱码 —— 对比一下 utf-8 和 utf-8-sig 的差别')


In [ ]:
# 读回来验证一下
with open(out_file, 'r', encoding='utf-8-sig', newline='') as f:
    rows = list(csv.reader(f))
print('表头：', rows[0])
print('前 3 行：', rows[1:4])
print('总行数（不含表头）：', len(rows) - 1)


---
## 七、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 给地名加"省/市"后缀 | 区域灰色，**不报错** | 用原始标识名 |
| 硬编码 `["areaTree"][0]` 但不确认结构 | `KeyError` / `IndexError` | 先逐层打印确认 |
| 颜色写成 `##0000FF` | 渲染异常 | 一个 `#` |
| 数值跨度大还用连续型 | 大部分区域同色 | 改用 `is_piecewise=True` |
| 读文件用了桌面绝对路径 | 换电脑就崩 | 用相对的定位函数 |
| CSV 导出中文乱码 | Excel 打开是一堆问号 | 用 `encoding='utf-8-sig'` |
| CSV 多出空行 | Windows 下换行符被写两次 | 加 `newline=''` |
| 忘了 `is_map_symbol_show=False` | 地图上多出一堆红点 | 按需关闭 |
| 用 `data` 字典而不是列表 | 类型错误 | 必须是 `(地名, 数值)` 列表 |

---

## 八、动手练习

**练习 1**：把 `pieces` 的分档改成你自己的想法（比如按 3 档分），
观察地图的视觉效果变化。

**练习 2**：算出所有省份的确诊总人数，用 `print` 输出；
再算出确诊人数前 5 名占全国的比例。

**练习 3**：用 `data/地图数据/疫情统计.csv` 画一张"治愈率"（治愈/确诊）的地图。
提示：治愈率是小数，`pieces` 里可以用 `{'min': 0, 'max': 0.5, ...}`。

**练习 4**：把地图和 TOP10 条形图的结果写进一个 Markdown 报告文件
（用 `with open(..., 'w')` 拼字符串）。

**练习 5**：改造 `data_path()`，如果找不到就**自动生成**一份示例数据，
让 notebook 在最干净的环境里也能跑起来。

---

**答案思路（先自己做）**

```python
# 练习 2
total = sum(v for _, v in data_list)
print('总人数：', total)

top5 = sorted(data_list, key=lambda x: x[1], reverse=True)[:5]
top5_sum = sum(v for _, v in top5)
print(f'前 5 名占比：{top5_sum / total:.1%}')

# 练习 3
import csv
with open(data_path('地图数据/疫情统计.csv'), encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

rate_data = [
    (r['省份'], round(int(r['治愈人数']) / max(int(r['确诊人数']), 1), 3))
    for r in rows
]
m = Map()
m.add('治愈率', rate_data, 'china')
m.set_global_opts(
    title_opts=opts.TitleOpts(title='各省治愈率'),
    visualmap_opts=opts.VisualMapOpts(is_piecewise=True, pieces=[
        {'min': 0, 'max': 0.5, 'label': '0-50%', 'color': '#FFE4E1'},
        {'min': 0.5, 'max': 0.8, 'label': '50%-80%', 'color': '#FF9999'},
        {'min': 0.8, 'label': '80% 以上', 'color': '#CC0000'},
    ]),
)
m.render('治愈率地图.html')
```
